# Demo of network estimation and densification

Steps:
1. Form a network with network points
2. Estimate point ambiguities of each network point.
3. Estimate point ambiguities and unwrapped phases of densification points.

The dataset used in this demo is a STM dataset over Amsterdam, consisting of 173 PS points. The data is available [here](https://zenodo.org/records/15324181).

In [ ]:
import xarray as xr
import numpy as np
from matplotlib import pyplot as plt
import matplotlib.colors as plc

from depsi.arc_estimation import periodogram
from depsi.classification import network_stm_selection
from depsi.network import (
    form_network,
    spatial_integration,
)
from depsi.densification import densification

In [ ]:
# Load test data
stm = xr.open_zarr('../../../data/stm_amsterdam_173p.zarr')

# Remove the mother epoch
# Mother image is with all h2ph values as 0
idx_non_mother = np.squeeze(np.where(stm['h2ph_values'].mean(axis=0).values != 0)) 
stm = stm.isel(time = idx_non_mother)

# For debugging, shorten the time series
stm = stm.isel(time=slice(0, 30))
stm

In [ ]:
# Select network points
# Make points sparser
stm_network_pnts = network_stm_selection(
                stm,
                min_dist=20,
                azimuth_spacing=20,
                range_spacing=5,
                sortby_var="nmad_full",
            )

# Assign wavelength as an attribute
WAVELENGTH = 0.055465763  # Sentinel-1 wavelength in meters 
stm_network_pnts = stm_network_pnts.assign_attrs({"wavelength": WAVELENGTH})

stm_network_pnts

In [ ]:
# Form network arcs
stm_network_arcs = form_network(
                stm_network_pnts,
                key_phase='sd_phase',
                key_h2ph='h2ph_values',
                key_Btemp='years',
                max_length=0.001,  # max length is dummy length based on lat/lon
            )
stm_network_arcs

In [ ]:
# arc estimation using periodogram
_, ambiguities, _, _, ens_coh = periodogram(
                stm_network_arcs,
                key_dphase="d_phase",
                key_h2ph="h2ph",
                key_Btemp="Btemp",
            )
stm_network_arcs["ambiguities"] = ambiguities
stm_network_arcs["temp_coh"] = ens_coh

stm_network_arcs = stm_network_arcs.compute()

In [ ]:
# Visualize estimated arcs
xx = np.stack([stm_network_pnts.isel(space = stm_network_arcs['source'])['lon'].values,
               stm_network_pnts.isel(space = stm_network_arcs['target'])['lon'].values]).T
yy = np.stack([stm_network_pnts.isel(space = stm_network_arcs['source'])['lat'].values,
               stm_network_pnts.isel(space = stm_network_arcs['target'])['lat'].values]).T
# Visualize created arcs
fig, ax = plt.subplots()
cmap = plt.cm.rainbow
norm = plc.Normalize(vmin=0, vmax=1.0)
mean_nmad = np.abs(stm_network_arcs["temp_coh"].data)
for i in range(stm_network_arcs.sizes["space"]):
    ax.plot(xx[i], yy[i], color=cmap(norm(mean_nmad[i])), linewidth=0.5)
plt.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, label="temp_coh")

In [ ]:
# Integrate ambiguities from arcs to points
stm_arcs_output, stm_pnts_output= spatial_integration(
    stm_network_pnts,
    stm_network_arcs,
    key_arc_quality = "temp_coh",
    threshold_arc_quality = 0.5,
    idx_refpnt = None,
)
stm_pnts_output

In [ ]:
# visualize estimated point ambiguities
plt.imshow(stm_pnts_output["ambiguities"].values)

In [ ]:
stm_densified = densification(stm, stm_pnts_output)
stm_densified

In [ ]:
# visualize densified ambiguities
plt.imshow(stm_densified["ambiguities"].values)

In [ ]:
# Visualize estimated arcs
xx = np.stack([stm_network_pnts.isel(space = stm_network_arcs['source'])['lon'].values,
               stm_network_pnts.isel(space = stm_network_arcs['target'])['lon'].values]).T
yy = np.stack([stm_network_pnts.isel(space = stm_network_arcs['source'])['lat'].values,
               stm_network_pnts.isel(space = stm_network_arcs['target'])['lat'].values]).T
# Visualize created arcs
fig, ax = plt.subplots()
cmap = plt.cm.rainbow
norm = plc.Normalize(vmin=0, vmax=1.0)
mean_nmad = np.abs(stm_network_arcs["temp_coh"].data)
for i in range(stm_network_arcs.sizes["space"]):
    ax.plot(xx[i], yy[i], color=cmap(norm(mean_nmad[i])), linewidth=0.5)
ax.scatter(stm_densified['lon'].values, stm_densified['lat'].values, c='blue', s=2, label='Densified Points')
ax.scatter(stm_network_pnts['lon'].values, stm_network_pnts['lat'].values, c='red', s=5, label='Network Points')
ax.legend()
plt.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, label="temp_coh")